In [ ]:
import os
import random
import requests
from PIL import Image
import ipywidgets as widgets
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

# Configure Stable Diffusion API via Hugging Face Inference Framework
# Get a free API token at: huggingface.co/settings/tokens
HF_TOKEN = os.getenv("HF_TOKEN", "your_huggingface_api")
MODEL_ID = "stabilityai/stable-diffusion-2-1"
API_URL = f"https://api-inference.huggingface.co/models/{MODEL_ID}"

print(f"Stable Diffusion API Gateway Target Configured: {MODEL_ID}")

Stable Diffusion API Gateway Target Configured: stabilityai/stable-diffusion-2-1


In [2]:
def generate_and_stream_image_studio(
    prompt: str, 
    aspect_ratio: str, 
    resolution_scale: str, 
    steps: int, 
    seed: int, 
    output_filename: str = "studio_artwork.png"
):
    """
    Executes the complete 6-stage production framework over the Stable Diffusion API:
    1. Formulates parameterized options payload ensuring dimensions are multiples of 8.
    2. Implements connection pooling with automatic backoff retries.
    3. Handles timeouts (3.05s connect / 60s read) and security blocks gracefully.
    4. Pipes response using a memory-safe 64KiB chunk loop directly to disk storage.
    """
    # Stage 1: Prompt Payload Formulation (Valid SD dimensions must be multiples of 8)
    base_dim = 768 if resolution_scale == "Standard (768p)" else 1024
    
    if aspect_ratio == "1:1 Square":
        width, height = base_dim, base_dim
    elif aspect_ratio == "16:9 Landscape":
        width, height = base_dim, int(base_dim * (9 / 16)) // 8 * 8
    elif aspect_ratio == "9:16 Portrait":
        width, height = int(base_dim * (9 / 16)) // 8 * 8, base_dim
    elif aspect_ratio == "4:3 Classic":
        width, height = base_dim, int(base_dim * (3 / 4)) // 8 * 8
    
    chosen_seed = seed if seed != -1 else random.randint(1, 99999999)
    print(f"\n[Stage 1] Formulation Payload Applied: Shape Constraint {width}x{height} | Seed: {chosen_seed}")
    
    headers = {
        "Authorization": f"Bearer {HF_TOKEN}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "inputs": prompt,
        "parameters": {
            "width": width,
            "height": height,
            "num_inference_steps": min(steps, 50), # Stable Diffusion scales effectively up to 50 steps
            "seed": chosen_seed
        }
    }
    
    # Stage 2 & 3: Network API Gateway Integration & Dual Security Gates
    print("[Stage 2 & 3] Routing payload safely via resilient security gateways...")
    
    retries = Retry(
        total=3, 
        backoff_factor=2, # Exponential backoff factor (2s, 4s, 8s) to handle busy endpoints
        status_forcelist=[500, 502, 503, 504],
        raise_on_status=False
    )
    
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retries))
    
    try:
        # Activating stream=True to prevent raw image bytes from stacking up inside RAM (image_c8487c.jpg)
        # Using strict (3.05s connect, 60s read) timeout limits mapped in image_c848df.jpg
        response = session.post(API_URL, headers=headers, json=payload, stream=True, timeout=(3.05, 60))
        
        # Intercepting dual security gate rejections and handling error traces matching image_c84824.jpg
        if response.status_code in [400, 422]:
            error_msg = response.text.lower()
            if "sentinel_block" in error_msg or "content_policy_violation" in error_msg:
                print("[GATE 1 REJECTION] Pre-Generation Input Filter Blocked This Request. Compute Impact: 0 compute cost incurred. Instant rejection.")
                return None
            elif "moderation_blocked" in error_msg or "finish_reason=filter" in error_msg:
                print("[GATE 2 REJECTION] Post-Generation Output Filter Blocked Content. Compute Impact: Compute cost fully incurred.")
                return None
            else:
                print(f"[API Error] Bad payload parameters or model initialization: {response.text}")
                return None
        elif response.status_code in [401, 403]:
            print("[Gateway Error] Authentication failed. Please verify your HF_TOKEN value in Cell 2.")
            return None
            
        response.raise_for_status()
        
    except requests.exceptions.ConnectionError as conn_err:
        print(f"[Gateway Error] Connection pool refused by the endpoint.\nDetails: {conn_err}")
        return None
    except requests.exceptions.Timeout:
        print("[Gateway Error] Core network gateway timeout constraint exceeded (3.05s Connection / 60s Read rule).")
        return None
    except Exception as api_err:
        print(f"[API Error] Critical gateway failure exception: {api_err}")
        return None

    # Stage 4 & 5: Memory-Safe Transport Protocol and Integrity Check
    print("[Stage 4] Streaming payload directly to local disk via memory-safe 64KiB chunk pipeline...")
    try:
        if os.path.exists(output_filename):
            os.remove(output_filename)
            
        with open(output_filename, 'wb') as f:
            # Rule 3 from image_c8487c.jpg: Write sequentially using iter_content(chunk_size=65536)
            for chunk in response.iter_content(chunk_size=65536):
                if chunk:
                    f.write(chunk)
                    
        file_size = os.path.getsize(output_filename)
        if file_size < 2000:
            print(f"[Storage Error] Stream error: file payload too small ({file_size} bytes). Endpoint may be returning a json error message.")
            # Read and print the content if it's text data instead of an image bytes payload
            with open(output_filename, 'r', errors='ignore') as err_file:
                print(f"Server Response: {err_file.read()[:200]}")
            return None
            
        print(f"[Stage 5] Transport written safely ({file_size} bytes). Verifying canvas pixel decoding matrix...")
        
        # Forced .load() validation ensuring image pixel verification (image_c848df.jpg Stage 5)
        with Image.open(output_filename) as verified_img:
            verified_img.load()
            print("[Stage 5 Verification Passed] Stable Diffusion image payload is completely stable and renderable.")
            return verified_img
            
    except Exception as stream_err:
        print(f"[Storage/Verification Error] Asset generation failed downstream checks: {stream_err}")
    return None

In [ ]:
# 1. Define interactive UI layout components cleanly
prompt_widget = widgets.Textarea(
    value='A highly detailed, cinematic digital rendering of a futuristic laboratory inside a greenhouse, cyberpunk elements',
    placeholder='Type your design prompt instructions...',
    description='Prompt:',
    layout=widgets.Layout(width='90%', height='80px')
)

ratio_widget = widgets.Dropdown(
    options=['1:1 Square', '16:9 Landscape', '9:16 Portrait', '4:3 Classic'],
    value='16:9 Landscape',
    description='Aspect Ratio:',
)

scale_widget = widgets.Dropdown(
    options=['Standard (768p)', 'High (1024p)'],
    value='Standard (768p)',
    description='Res Quality:',
)

steps_widget = widgets.IntSlider(
    value=30,
    min=15,
    max=50,
    step=5,
    description='Steps (Quality):',
)

seed_widget = widgets.IntText(
    value=-1,
    description='Seed (-1 Rand):',
)

generate_btn = widgets.Button(
    description='Orchestrate Studio Generation',
    button_style='info',
    tooltip='Click to safely call pipeline and process input configuration options',
    icon='paint-brush',
    layout=widgets.Layout(width='40%', height='40px')
)

# Explicitly define output frame region
output_area = widgets.Output()

# 2. Link execution pipeline logic directly inside the widget button action event handler
def on_click_orchestrate(b):
    with output_area:
        output_area.clear_output()
        
        chosen_prompt = prompt_widget.value
        chosen_ratio = ratio_widget.value
        chosen_scale = scale_widget.value
        chosen_steps = steps_widget.value
        chosen_seed = seed_widget.value
        
        print("Launching Stable Diffusion studio generation pipeline...")
        print(f"User Prompt Input: '{chosen_prompt}'")
        
        generate_btn.disabled = True
        try:
            # Calls the verified 64KiB chunk streaming function over Stable Diffusion
            img_asset = generate_and_stream_image_studio(
                prompt=chosen_prompt,
                aspect_ratio=chosen_ratio,
                resolution_scale=chosen_scale,
                steps=chosen_steps,
                seed=chosen_seed,
                output_filename="stable_diffusion_artwork.png"
            )
            
            if img_asset is not None:
                print("\nRendering visual asset canvas matrix below:")
                display(img_asset)
                
        except Exception as pipeline_err:
            print(f"\n[Execution Failure] Runtime error: {pipeline_err}")
        finally:
            generate_btn.disabled = False

# Attach the click event observer mapping to the button instance
generate_btn.on_click(on_click_orchestrate)

# 3. Assemble and display the application control interface blocks
control_panel = widgets.VBox([
    prompt_widget,
    widgets.HBox([ratio_widget, scale_widget]),
    widgets.HBox([steps_widget, seed_widget]),
    widgets.Label(value=""),
    generate_btn,
    output_area
])

display(control_panel)